# Business Case — ONG World Vision
## Identification des pays prioritaires pour l'accès à l'eau potable

**Auteur :** Mégane La Rocca  
**Formation :** Data Analyst RNCP6 — Promotion Vert 11-2025  
**Date :** 21 avril 2026  
**Durée de l'exercice :** 8h

## 1. Contexte

L'ONG **World Vision** intervient à l'échelle mondiale pour fournir un accès à l'eau potable : création de services, modernisation des infrastructures, accompagnement des politiques publiques.

Jusqu'à présent, le choix des zones d'intervention s'est fait de manière **artisanale**, sans approche data-driven. L'ONG souhaite aujourd'hui **maximiser l'impact** de ses actions en s'appuyant sur les données.

## 2. Problématique

> **Quels sont les pays prioritaires où l'ONG World Vision doit concentrer ses interventions pour maximiser son impact sur l'accès à l'eau potable ?**

## 3. Objectif du livrable

Je dois produire un dashboard interactif offrant **3 vues complémentaires** :
- **Mondiale** : panorama global et identification des zones critiques
- **Régionale** : analyse comparée des continents / régions
- **Nationale** : zoom pays pour les décisions opérationnelles

## 4. Hypothèses de travail

→ Un pays prioritaire combine plusieurs critères : **forte mortalité liée à l'eau**, **faible accès à l'eau potable**, **population importante** (impact potentiel), et **stabilité politique suffisante** (faisabilité de l'intervention).

→ Les inégalités **urbain / rural** et **hommes / femmes** peuvent révéler des leviers d'action ciblés.

→ La construction d'un **score de priorité composite** permettra de hiérarchiser les pays de manière objective et argumentable.

## 5. Démarche

1. **Collecte** : chargement des 5 datasets fournis
2. **Exploration** : audit qualité, structure, relations entre datasets
3. **Préparation** : nettoyage, jointures, création de variables dérivées
4. **Analyse** : réponse aux questions métier (mortalité, politique, urbain/rural, genre)
5. **Modélisation** : construction d'un score de priorité
6. **Visualisation** : graphiques, carte, tableau croisé
7. **Conclusion** : recommandations pour les investisseurs

## 6. Datasets utilisés

- `df_population` : population mondiale par pays (2000-2018)
- `df_basicsafewater` : accès eau potable rural/urbain par pays (2000-2017)
- `df_mortality` : taux de mortalité lié à l'eau non potable (2016)
- `df_political` : stabilité politique par pays (2000-2017)
- `df_region_country` : région d'appartenance de chaque pays

## 7. Import des bibliothèques

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print("Bibliothèques importées avec succès")

Bibliothèques importées avec succès


## 8. Chargement des données

je réalise le chargement des données en mettant directement chaque id pour chaque chargement. Je trouve cela plus fluide que réassigner la variable 'id' à chaque fois. 

In [3]:
def get_file(id: str) -> pd.DataFrame:
    """Je télécharge un fichier CSV depuis Google Drive via son ID."""
    url = f'https://drive.google.com/uc?id={id}'
    return pd.read_csv(url)

df_population = get_file('1DEOHEXtyZC-FjQOoEzVUA4ZCbt4_AYFK')
df_basicsafewater = get_file('1X03LbUPaQcoiBdxgJvDcNqGIMfHhsxSz')
df_mortality = get_file('1tUVfNHS6MEarPR9oZCayCBNw-V8-fc1n')
df_political = get_file('1cpVJg3QFHmZerPbCmjJteGQYV8iprVYE')
df_region_country = get_file('1y_jygG2YhuKJ5BoWoW4M394XSHYas0Jf')

print("Chargement des 5 datasets terminé")

Chargement des 5 datasets terminé


## 9. Vue d'ensmble des datasets

In [ ]:
Afin d'optimiser le temps d'exploration, je vais:
→ Charger les 5 DataFrames dans un dictionnaire
→ Faire l'audit en boucle sur les 5 d'un coup
→ 5x moins de code 

In [7]:
# Je charge les 5 datasets dans un dictionnaire
dfs = {
    'population': df_population,
    'basicsafewater': df_basicsafewater,
    'mortality': df_mortality,
    'political': df_political,
    'region_country': df_region_country
}

# J'audite tous les datasets d'un coup
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"DATASET : {name}")
    print(f"{'='*60}")
    print(f"Shape : {df.shape}")
    print(f"Colonnes : {df.columns.tolist()}")
    print(f"Doublons : {df.duplicated().sum()}")
    print(f"NaN totaux : {df.isnull().sum().sum()}")
    print(f"\nAperçu :")
    print(df.head(3))


DATASET : population
Shape : (20914, 4)
Colonnes : ['Country', 'Granularity', 'Year', 'Population']
Doublons : 0
NaN totaux : 0

Aperçu :
       Country Granularity  Year  Population
0  Afghanistan       Total  2000   20779.953
1  Afghanistan        Male  2000   10689.508
2  Afghanistan      Female  2000   10090.449

DATASET : basicsafewater
Shape : (10476, 5)
Colonnes : ['Year', 'Country', 'Granularity', 'Population using at least basic drinking-water services (%)', 'Population using safely managed drinking-water services (%)']
Doublons : 0
NaN totaux : 8251

Aperçu :
   Year      Country Granularity  \
0  2000  Afghanistan       Rural   
1  2000  Afghanistan       Total   
2  2000  Afghanistan       Urban   

   Population using at least basic drinking-water services (%)  \
0                                           21.61913             
1                                           27.77190             
2                                           49.48745             

   Population

## Synthèse de l'audit initial

### Volumétrie et structure

→ **5 datasets chargés** avec succès, pas de doublons détectés  
→ **population** : 20 914 lignes × 4 colonnes — propre, 0 NaN  
→ **political** : 3 526 lignes × 4 colonnes — propre, 0 NaN  
→ **region_country** : 194 lignes × 2 colonnes — table de référence, 0 NaN  
→ **basicsafewater** : 10 476 lignes × 5 colonnes — **8 251 NaN à investiguer**  
→ **mortality** : 549 lignes × 5 colonnes — **366 NaN à investiguer**  

### Points d'attention identifiés

→ → La colonne **"safely managed drinking-water"** concentre probablement la majorité des NaN de `basicsafewater` — indicateur avec **trop de valeurs manquantes** pour être exploité 
→ Dans `mortality`, la colonne **`WASH deaths`** a des NaN quand `Granularity` ≠ Total — à vérifier  
→ Les datasets ont tous une colonne **`Country`** et souvent **`Granularity`** → jointures et filtrages possibles  
→ `region_country` utilise des noms de colonnes différents (`COUNTRY (DISPLAY)`) → harmonisation nécessaire avant jointure  

### Prochaine étape

→ **Zoom ciblé** sur `basicsafewater` et `mortality` pour identifier précisément les colonnes exploitables  
→ Décision à prendre : **conserver ou abandonner** la colonne "safely managed drinking-water"

## 10. Exploration complémentaire: types, sats, modalités

Ce que je veux savoir maintenant pour pouvoir prendre une décision métier:
→ Types de données (certaines colonnes doivent être converties)
→ Stats descriptives des variables numériques (ordre de grandeur, outliers)
→ Zoom sur les NaN des 2 datasets à problèmes

In [8]:
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"DATASET : {name}")
    print(f"{'='*60}")
    print("\n--- INFO ---")
    df.info()
    print("\n--- DESCRIBE ---")
    print(df.describe(include='all'))


DATASET : population

--- INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 20914 entries, 0 to 20913
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Country      20914 non-null  str    
 1   Granularity  20914 non-null  str    
 2   Year         20914 non-null  int64  
 3   Population   20914 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 973.9 KB

--- DESCRIBE ---
            Country Granularity          Year    Population
count         20914       20914  20914.000000  2.091400e+04
unique          239           5           NaN           NaN
top     Afghanistan       Total           NaN           NaN
freq             95        4430           NaN           NaN
mean            NaN         NaN   2009.046715  2.253164e+04
std             NaN         NaN      5.479195  1.000169e+05
min             NaN         NaN   2000.000000  0.000000e+00
25%             NaN         NaN   2004.000000  3.483462e+

In [9]:
# Maintenant que j'ai plus de détail, je calcule le % de remplissage des colonnes potentiellement problématiques

print("=== basicsafewater ===")
for col in df_basicsafewater.columns:
    taux = df_basicsafewater[col].notnull().mean() * 100
    print(f"{col} : {taux:.1f}% rempli")

print("\n=== mortality ===")
for col in df_mortality.columns:
    taux = df_mortality[col].notnull().mean() * 100
    print(f"{col} : {taux:.1f}% rempli")

=== basicsafewater ===
Year : 100.0% rempli
Country : 100.0% rempli
Granularity : 100.0% rempli
Population using at least basic drinking-water services (%) : 89.9% rempli
Population using safely managed drinking-water services (%) : 31.4% rempli

=== mortality ===
Year : 100.0% rempli
Country : 100.0% rempli
Granularity : 100.0% rempli
Mortality rate attributed to exposure to unsafe WASH services : 100.0% rempli
WASH deaths : 33.3% rempli


markdown## Décisions sur les colonnes et le périmètre

### Taux de remplissage des colonnes lacunaires
Le code ci-dessus me confirme les colonnes exploitables :
→ **basic drinking-water services (~90%)** : conservée, indicateur principal d'accès à l'eau  
→ **safely managed drinking-water services (~31%)** : abandonnée, trop peu de données fiables  
→ **Mortality rate (100%)** : conservée  
→ **WASH deaths (~33%)** : renseigné uniquement pour `Granularity = Total`, conservée pour analyses globales uniquement  

### Pas d'imputation
Je choisis de **ne pas imputer** les valeurs manquantes : ce sont des données officielles OMS, toute imputation biaiserait l'analyse. Je travaille sur les données disponibles et reste transparente sur les pays absents.

### Types de données
Les types sont cohérents avec la nature des données (str pour catégoriel, int/float pour numérique). **Aucune conversion nécessaire.**

### Périmètre : pays communs
Les datasets couvrent entre 183 et 239 pays. Pour les analyses croisées, je travaillerai sur l'**intersection des pays communs**, via des jointures sur la clé `Country`. Le volume final sera donc **réduit** — Cela garantit une analyse cohérente et de qualité. 

## 11. Nettoyage et harmonisation

### Objectif
Préparer les datasets pour les jointures : renommer les colonnes, supprimer les colonnes inexploitables, standardiser les noms pour faciliter les croisements.

### Actions prévues
→ **basicsafewater** : drop de la colonne `safely managed drinking-water` (31% rempli), renommage de `basic drinking-water` en nom court  
→ **mortality** : renommage de `Mortality rate attributed to exposure to unsafe WASH services` en nom court  
→ **region_country** : renommage de `REGION (DISPLAY)` → `Region` et `COUNTRY (DISPLAY)` → `Country`  
→ **Vérification** : affichage des colonnes finales pour validation

In [ ]:
# basicsafewater : je drop la colonne trop lacunaire et renomme
df_basicsafewater = df_basicsafewater.drop(columns=['Population using safely managed drinking-water services (%)'])
df_basicsafewater = df_basicsafewater.rename(columns={
    'Population using at least basic drinking-water services (%)': 'basic_water_access_pct'
})

# mortality : je renomme la colonne cible
df_mortality = df_mortality.rename(columns={
    'Mortality rate attributed to exposure to unsafe WASH services': 'mortality_rate'
})

# region_country : j'harmonise les noms de colonnes
df_region_country = df_region_country.rename(columns={
    'REGION (DISPLAY)': 'Region',
    'COUNTRY (DISPLAY)': 'Country'
})



In [12]:
# Je reconstruis le dictionnaire pour qu'il pointe vers les versions à jour
dfs = {
    'population': df_population,
    'basicsafewater': df_basicsafewater,
    'mortality': df_mortality,
    'political': df_political,
    'region_country': df_region_country
}

# Je vérifie à nouveau
for name, df in dfs.items():
    print(f"{name} : {df.columns.tolist()}")

population : ['Country', 'Granularity', 'Year', 'Population']
basicsafewater : ['Year', 'Country', 'Granularity', 'basic_water_access_pct']
mortality : ['Year', 'Country', 'Granularity', 'mortality_rate', 'WASH deaths']
political : ['Country', 'Year', 'Political_Stability', 'Granularity']
region_country : ['Region', 'Country']


## Construction des datasets consolidés

### Objectif
je fais le choix après reflexion de créer **deux datasets** complémentaires :
→ **`df_priorite`** : snapshot des dernières données disponibles par pays → servira au **score de priorité** que je compte construire 
→ **`df_temporel`** : données multi-années → servira aux **graphiques d'évolution** 

### Clés de jointure
→ `Country` pour toutes les jointures  
→ `Year` en complément pour le dataset temporel  

### Type de jointure
→ **`inner`** : je conserve uniquement les pays présents dans tous les datasets, pour une analyse cohérente  

### Étape préalable
Filtrer sur `Granularity = 'Total'` pour la consolidation principale (les analyses Rural/Urban et Male/Female se feront séparément sur les datasets d'origine).

### Étape 1 : filtrer sur Total

In [13]:
# Je filtre chaque dataset sur Granularity = Total
pop_total = df_population[df_population['Granularity'] == 'Total'].copy()
water_total = df_basicsafewater[df_basicsafewater['Granularity'] == 'Total'].copy()
mortality_total = df_mortality[df_mortality['Granularity'] == 'Total'].copy()
political_total = df_political[df_political['Granularity'] == 'Total'].copy()

# Je vérifie les dimensions
print("population Total :", pop_total.shape)
print("water Total :", water_total.shape)
print("mortality Total :", mortality_total.shape)
print("political Total :", political_total.shape)

population Total : (4430, 4)
water Total : (3492, 4)
mortality Total : (183, 5)
political Total : (3526, 4)


### Étape 2 : Construction du dataset temporel

Jointure inner sur Country + Year entre population, eau et politique. Mortalité traitée séparément (2016 uniquement).

In [14]:
# Je joins population + eau sur Country + Year
df_temporel = pop_total.merge(water_total, on=['Country', 'Year'], how='inner')

# J'ajoute la stabilité politique
df_temporel = df_temporel.merge(political_total, on=['Country', 'Year'], how='inner')

# Je nettoie les colonnes Granularity en double (toutes à "Total")
df_temporel = df_temporel.drop(columns=[c for c in df_temporel.columns if 'Granularity' in c])

# J'ajoute la région (jointure sur Country uniquement)
df_temporel = df_temporel.merge(df_region_country, on='Country', how='inner')

print("df_temporel :", df_temporel.shape)
print("Colonnes :", df_temporel.columns.tolist())
df_temporel.head()

df_temporel : (3152, 6)
Colonnes : ['Country', 'Year', 'Population', 'basic_water_access_pct', 'Political_Stability', 'Region']


,Country,Year,Population,basic_water_access_pct,Political_Stability,Region
0,Afghanistan,2000,20779.953,27.77190,-2.44,Eastern Mediterranean
1,Afghanistan,2002,22600.770,29.90076,-2.04,Eastern Mediterranean
2,Afghanistan,2003,23680.871,32.00507,-2.20,Eastern Mediterranean
3,Afghanistan,2004,24726.684,34.12623,-2.30,Eastern Mediterranean
4,Afghanistan,2005,25654.277,36.26526,-2.07,Eastern Mediterranean


## Audit de `df_temporel`

### Objectif
Vérifier la qualité du dataset consolidé avant de construire `df_priorite` : doublons, valeurs manquantes, couverture temporelle et géographique.

### Critères vérifiés
→ **Shape** : dimensions finales après jointures  
→ **Doublons** : aucune ligne redondante  
→ **NaN** : valeurs manquantes sur les colonnes clés (notamment `basic_water_access_pct`)  
→ **Pays uniques** : périmètre géographique après intersection  
→ **Années couvertes** : continuité temporelle  
→ **Régions** : vérification des 6 zones OMS attendues

In [15]:
print("Shape :", df_temporel.shape)
print("\nDoublons :", df_temporel.duplicated().sum())
print("\nNaN par colonne :")
print(df_temporel.isnull().sum())
print("\nNombre de pays uniques :", df_temporel['Country'].nunique())
print("Années couvertes :", sorted(df_temporel['Year'].unique()))
print("Régions :", df_temporel['Region'].unique())

Shape : (3152, 6)

Doublons : 0

NaN par colonne :
Country                    0
Year                       0
Population                 0
basic_water_access_pct    20
Political_Stability        0
Region                     0
dtype: int64

Nombre de pays uniques : 190
Années couvertes : [np.int64(2000), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
Régions : <ArrowStringArray>
['Eastern Mediterranean',                'Europe',                'Africa',
              'Americas',       'Western Pacific',       'South-East Asia']
Length: 6, dtype: str


## Audit de `df_temporel` — Interprétation

### Points validés
→ **0 doublon**, dataset propre  
→ **190 pays** couverts (intersection des 5 datasets)  
→ **6 régions OMS** représentées  
→ **Population, Political_Stability, Region** : 100% remplis  

### Points d'attention
→ **20 NaN sur `basic_water_access_pct`** (0.6% du dataset) → à investiguer  
→ **Année 2001 absente** : `df_political` n'a pas de données sur 2001, la jointure inner l'a donc exclue. Impact minime (18 années sur 19).  

### Zoom sur les NaN
J'identifie les pays/années concernés avant de décider du traitement (imputation, drop, conservation).

In [16]:
# Je regarde quels pays/années ont des NaN sur l'accès à l'eau
nan_water = df_temporel[df_temporel['basic_water_access_pct'].isnull()]
print("Pays concernés :", nan_water['Country'].unique())
print("\nDétail :")
print(nan_water[['Country', 'Year']].to_string())

Pays concernés : <ArrowStringArray>
[               'Argentina', 'Central African Republic',
                 'Dominica',                  'Eritrea',
                     'Oman',                   'Poland',
        'Republic of Korea',    'Saint Kitts and Nevis',
              'Timor-Leste', 'United States of America']
Length: 10, dtype: str

Détail :
                       Country  Year
118                  Argentina  2017
560   Central African Republic  2017
833                   Dominica  2016
834                   Dominica  2017
936                    Eritrea  2017
2099                      Oman  2000
2233                    Poland  2000
2234                    Poland  2002
2235                    Poland  2003
2236                    Poland  2004
2284         Republic of Korea  2000
2380     Saint Kitts and Nevis  2014
2381     Saint Kitts and Nevis  2015
2382     Saint Kitts and Nevis  2016
2383     Saint Kitts and Nevis  2017
2782               Timor-Leste  2000
2999  United Stat

#Drop des 20 NaN:

→ 10 pays concernés, 20 lignes sur 3152 (0.6%)
→ Données manquantes en bordure temporelle (début 2000-2004 ou fin 2016-2017)
→  Une imputation par interpolation temporelle aurait été envisageable mais aurait introduit une incertitude sur des pays stratégiques comme les USA ou l'Argentine. Cela aurait fabriquer de la donnée pour des pays stratégiques (USA, Pologne, Corée du Sud, Argentine) et aurait introduit un biais dans l'analyse.
→ Drop = perte minime, intégrité maximale

In [17]:
# Je supprime les 20 lignes avec NaN sur basic_water_access_pct
df_temporel = df_temporel.dropna(subset=['basic_water_access_pct']).reset_index(drop=True)

# Je vérifie
print("Shape après drop :", df_temporel.shape)
print("NaN restants :", df_temporel.isnull().sum().sum())

Shape après drop : (3132, 6)
NaN restants : 0


## Audit final de `df_temporel`

### État après nettoyage
→ **Shape finale** : 3132 lignes × 6 colonnes  
→ **0 doublon, 0 NaN**  
→ **190 pays** sur **17 années** (2000, 2002-2017)  
→ **6 régions OMS** représentées  

### Décisions de traitement
→ **Drop des 20 NaN** sur `basic_water_access_pct` : 10 pays concernés (Argentine, RCA, Pologne, USA, Corée du Sud…) avec données manquantes en bordure temporelle. Impact : 0.6% du dataset.  
→ **Pas d'imputation** : fabriquer des valeurs pour des pays stratégiques aurait biaisé les analyses.  
→ **Année 2001 absente** : conséquence de la jointure inner avec `df_political` qui ne la couvrait pas. Aucun impact analytique.  

### Validation
→ Dataset propre, prêt pour la construction de `df_priorite` et les analyses.

## Etape 3 : construction de `df_priorite` (snapshot)

### Objectif
Dataset de référence pour le calcul du score de priorité : 1 ligne par pays avec les **dernières données disponibles**.

### Démarche
→ Je prends la **dernière année disponible** par pays sur `df_temporel` (2017 pour la plupart)  
→ J'ajoute la **mortalité 2016** (seule année disponible)  
→ Résultat : 1 ligne = 1 pays = photo actuelle de la situation

In [18]:
# Je prends la dernière année disponible par pays
df_priorite = df_temporel.sort_values('Year').groupby('Country').tail(1).reset_index(drop=True)

# J'ajoute la mortalité (jointure sur Country uniquement, car mortalité = 2016)
mortality_subset = mortality_total[['Country', 'mortality_rate', 'WASH deaths']]
df_priorite = df_priorite.merge(mortality_subset, on='Country', how='inner')

# Je vérifie
print("Shape :", df_priorite.shape)
print("Colonnes :", df_priorite.columns.tolist())
print("\nAnnées retenues :", df_priorite['Year'].value_counts().to_dict())
df_priorite.head()

Shape : (181, 8)
Colonnes : ['Country', 'Year', 'Population', 'basic_water_access_pct', 'Political_Stability', 'Region', 'mortality_rate', 'WASH deaths']

Années retenues : {2017: 178, 2016: 3}


,Country,Year,Population,basic_water_access_pct,Political_Stability,Region,mortality_rate,WASH deaths
0,Central African Republic,2016,4537.686,46.33376,-1.79,Africa,82.11312,3772.78400
1,Argentina,2016,43508.460,99.07838,0.20,Americas,0.36294,159.14110
2,Eritrea,2016,3376.557,51.84972,-0.66,Africa,45.56445,2257.55600
3,Liberia,2017,4702.226,72.94803,-0.36,Africa,41.54313,1916.72900
4,Lithuania,2017,2845.414,97.54207,0.78,Europe,0.08078,2.34934


#Audit de df_priorité:

In [19]:
print("Shape :", df_priorite.shape)
print("\nDoublons :", df_priorite.duplicated().sum())
print("\nNaN par colonne :")
print(df_priorite.isnull().sum())
print("\nNombre de pays uniques :", df_priorite['Country'].nunique())
print("Régions :", df_priorite['Region'].unique())
print("\nStats descriptives :")
print(df_priorite.describe())

Shape : (181, 8)

Doublons : 0

NaN par colonne :
Country                   0
Year                      0
Population                0
basic_water_access_pct    0
Political_Stability       0
Region                    0
mortality_rate            0
WASH deaths               0
dtype: int64

Nombre de pays uniques : 181
Régions : <ArrowStringArray>
[               'Africa',              'Americas',                'Europe',
       'Western Pacific', 'Eastern Mediterranean',       'South-East Asia']
Length: 6, dtype: str

Stats descriptives :
              Year    Population  basic_water_access_pct  Political_Stability  \
count   181.000000  1.810000e+02              181.000000           181.000000   
mean   2016.983425  3.358901e+04               86.901183            -0.135083   
std       0.128025  1.081495e+05               16.601380             0.973665   
min    2016.000000  9.542600e+01               38.700600            -2.940000   
25%    2017.000000  2.845414e+03               78.510

## Audit final de `df_priorite`

### Validation
→ **181 pays** uniques (1 ligne par pays)  
→ **0 doublon, 0 NaN** sur l'ensemble des colonnes  
→ **6 régions OMS** représentées  
→ Snapshot sur les dernières années disponibles (2016-2017)  

### Stats clés pour le score de priorité
→ **Accès à l'eau potable** : min 38.7 — max 100 → forte amplitude, bon pouvoir discriminant  
→ **Taux de mortalité** : min 0.006 — max 101 → écart très important, variable très discriminante  
→ **Stabilité politique** : min -2.94 — max +1.62 → échelle complète couverte  

### Conclusion
Dataset propre, cohérent, et suffisamment contrasté pour construire un score de priorité robuste..